# Xori Hori — LoRA training on Colab/Kaggle GPU

Обучаем `Qwen/Qwen2.5-0.5B-Instruct` через LoRA и загружаем адаптер в `Abobus2222228/Xoritg`.

**Перед запуском:** включи GPU. В Colab токен Hugging Face вводится в защищённом поле ниже; в Kaggle можно заменить его на Secret.

In [ ]:
!pip -q install -U transformers peft accelerate datasets huggingface_hub safetensors
import os, json, subprocess
from pathlib import Path
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('GPU не включён в Kaggle.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import login
import getpass
HF_TOKEN=getpass.getpass('Вставь Hugging Face token (input скрыт): ')
login(token=HF_TOKEN)
print('HF login OK')


In [ ]:
REPO_URL='https://github.com/xoristalin-dotcom/Xori-.git'
WORK=Path('/kaggle/working/Xori-')
if not WORK.exists(): subprocess.run(['git','clone','--depth','1',REPO_URL,str(WORK)],check=True)
os.chdir(WORK)
print('repo:',Path.cwd())


In [ ]:
pairs=[]
seed=Path('hori_sft_seed.jsonl')
if seed.exists():
    for line in seed.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r=json.loads(line)
            if r.get('user') and r.get('assistant'): pairs.append((r['user'],r['assistant']))
training=Path('hori_training.json')
if training.exists():
    data=json.loads(training.read_text(encoding='utf-8'))
    for r in data.get('examples',[]):
        if r.get('approved') and r.get('user') and r.get('assistant'): pairs.append((r['user'],r['assistant']))
seen=set(); clean=[]
for u,a in pairs:
    k=(u.strip(),a.strip())
    if k not in seen: seen.add(k); clean.append(k)
pairs=clean
print('training pairs:',len(pairs))
if len(pairs)<30: raise RuntimeError('Нужно минимум 30 примеров.')

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset
BASE='Qwen/Qwen2.5-0.5B-Instruct'
MAX_LEN=1024
SYSTEM='Ты — Хори Кёко из Horimiya. Отвечай по-русски естественно, прямо и по-человечески. Не копируй реплики из произведения. Не выдумывай личность или мысли собеседника. Не форсируй дружбу или романтику. Не управляй действиями собеседника.'
tokenizer=AutoTokenizer.from_pretrained(BASE,token=HF_TOKEN)
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
def build(u,a):
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':u}]
    prompt=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    full=prompt+a+tokenizer.eos_token
    enc=tokenizer(full,truncation=True,max_length=MAX_LEN,padding=False)
    p=tokenizer(prompt,add_special_tokens=False)['input_ids']
    n=min(len(p),len(enc['input_ids']))
    enc['labels']=[-100]*n+enc['input_ids'][n:]
    enc['labels']=enc['labels'][:len(enc['input_ids'])]
    return enc
rows=[build(u,a) for u,a in pairs]
dataset=Dataset.from_list(rows)
print(dataset)

In [ ]:
from transformers import AutoModelForCausalLM,TrainingArguments,Trainer
from transformers import DataCollatorForSeq2Seq
from peft import LoraConfig,get_peft_model
dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model=AutoModelForCausalLM.from_pretrained(BASE,token=HF_TOKEN,torch_dtype=dtype,device_map='auto')
cfg=LoraConfig(r=16,lora_alpha=32,lora_dropout=0.05,bias='none',task_type='CAUSAL_LM',target_modules=['q_proj','k_proj','v_proj','o_proj'])
model=get_peft_model(model,cfg)
model.print_trainable_parameters()
args=TrainingArguments(output_dir='xoritg_adapter',num_train_epochs=5,per_device_train_batch_size=2,gradient_accumulation_steps=8,learning_rate=2e-4,logging_steps=5,save_strategy='epoch',report_to='none',fp16=not torch.cuda.is_bf16_supported(),bf16=torch.cuda.is_bf16_supported(),gradient_checkpointing=True,optim='adamw_torch',remove_unused_columns=False)
collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,padding=True,label_pad_token_id=-100,return_tensors='pt')
trainer=Trainer(model=model,args=args,train_dataset=dataset,data_collator=collator)
trainer.train()

In [ ]:
REPO_ID='Abobus2222228/Xoritg'
trainer.save_model('xoritg_adapter')
tokenizer.save_pretrained('xoritg_adapter')
model.push_to_hub(REPO_ID,token=HF_TOKEN)
tokenizer.push_to_hub(REPO_ID,token=HF_TOKEN)
print('Uploaded:',REPO_ID)

In [ ]:
model.eval()
tests=['Привет, Хори. Как прошёл твой день?','Мне сегодня грустно.','Что ты сейчас делаешь?','Ты сразу доверяешь новым людям?']
for user_text in tests:
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':user_text}]
    prompt=tokenizer.apply_chat_template(msgs,tokenize=False,add_generation_prompt=True)
    x=tokenizer(prompt,return_tensors='pt').to(model.device)
    with torch.no_grad(): y=model.generate(**x,max_new_tokens=120,do_sample=True,temperature=0.72,top_p=0.9,repetition_penalty=1.08)
    reply=tokenizer.decode(y[0][x['input_ids'].shape[1]:],skip_special_tokens=True).strip()
    print('\nUSER:',user_text,'\nHORI:',reply)